# Performance Benchmarking: Building Reproducible Spark Benchmark Suites

**Module**: 10_performance_and_optimization  
**Notebook**: 07_performance_benchmarking.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How to design reproducible benchmarks in PySpark: warming up kernels, measuring wall-clock time, collecting Spark UI metrics, and comparing implementations.
- **Why does this topic exist?**: Optimization claims must be supported by empirical data, not guesswork.
- **What problem does it solve?**: Benchmarking un-warmed jobs or confusing client overhead with cluster compute time.
- **Where is it used in real-world Data Engineering?**: Validating migration of pipelines from on-premises to cloud.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Spark UI, plans, statistics |
| **Required depth** | Working knowledge: can design a reproducible benchmark and report honest results |
| **Data Engineer** | Build a benchmark that isolates one variable and reports distributions. |
| **Senior Data Engineer** | Design experiments that survive review and avoid misleading speedups. |
| **Spark specialist** | Reason about warmup, variance, caching, and workload representativeness. |
| **Next topic** | `11_fault_tolerance_and_reliability/01_lineage_and_recomputation.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A benchmark is an experiment, not a demo. The same query on the same data can produce very different numbers
depending on cache state, JIT warmup, cluster load, and data ordering. A credible benchmark fixes the data and
environment, changes one variable, runs enough repetitions, and reports a distribution rather than a single
best-case number.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Warmup** | Initial JIT/cache effects | Excluded from steady-state measurements |
| **Repetition** | Multiple runs | Captures variance and outliers |
| **Control** | Fixed environment | Same data, config, and cluster state |
| **Metric** | What to measure | Runtime, shuffle, spill, GC, cost |
| **Percentile** | Distribution summary | p50/p95 are more honest than the minimum |
| **Representativeness** | Realistic workload | Synthetic data should preserve skew and cardinality |

### How it works, step by step

1. Define the question and the variable to change.
2. Fix the data, cluster, and configuration.
3. Run a warmup and discard it.
4. Run multiple repetitions and record metrics.
5. Compare distributions and check for outliers.
6. Report the method and the uncertainty, not just the best number.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Benchmarking_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")

def benchmark_op(name, func, runs=3):
    # Warmup
    func()
    times = []
    for _ in range(runs):
        t0 = time.time()
        func()
        times.append(time.time() - t0)
    avg_time = sum(times) / len(times)
    print(f"Benchmark '{name}': Avg = {avg_time:.4f}s across {runs} runs (Min: {min(times):.4f}s)")
    return avg_time

benchmark_op("DataFrame Count", lambda: orders.count())
benchmark_op("DataFrame Distinct Status", lambda: orders.select("status").distinct().count())

---

## Experiment: Comparing Warmup vs Cold Run Latency

- **1. What are we testing?**: Demonstrate that the first run is slower due to class loading, JVM JIT compilation, and connection establishment.
- **2. What do we expect?**: First run is 2x slower than subsequent runs.

In [ ]:
t0 = time.time(); orders.groupBy("channel").count().count(); t_cold = time.time() - t0
t0 = time.time(); orders.groupBy("channel").count().count(); t_warm = time.time() - t0
print(f"Cold Run: {t_cold:.4f}s vs Warm Run: {t_warm:.4f}s")

**3. What happened?**: Cold run was noticeably slower.  
**4. Why did it happen?**: JVM JIT compilation, executor connection handshake, and classloading occurred on first run.  
**5. Practical Engineering Lesson**: Always run at least one warmup pass before recording benchmark measurements.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Validating a tuning change

- **Industry**: SaaS
- **Scenario**: A team wants to prove that a new partition count improves a nightly job.
- **Problem**: A single before/after run is noisy and may reflect cluster load rather than the change.
- **Why this concept is relevant**: A controlled benchmark with repeated runs and fixed data shows whether the improvement is real.
- **Implementation idea**: Run both configurations several times, alternate order, and compare p50/p95 runtime and shuffle.
- **Expected result**: A defensible recommendation with quantified variance.
- **Alternative**: Roll out the change gradually and compare production runs.
- **Trade-offs**: A canary rollout is realistic but slower and confounded by changing data volumes.

### Use Case 2: Vendor benchmark claim

- **Industry**: Financial services
- **Scenario**: A vendor claims a 5x speedup over the current platform.
- **Problem**: The benchmark uses a different dataset, scale, and query mix.
- **Why this concept is relevant**: A representative internal benchmark is needed before migration.
- **Implementation idea**: Replicate the top workloads on both platforms with the same data and measure end-to-end cost.
- **Expected result**: A migration decision based on the actual workload.
- **Alternative**: Run a pilot with a subset of production jobs.
- **Trade-offs**: A pilot is realistic but takes longer and may not cover edge cases.

---

## Business Value

- Credible performance claims.
- Better migration and procurement decisions.
- Less time wasted on noisy or misleading results.
- A culture of measurement rather than anecdote.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Fix the data and environment; change one variable.
- Discard warmup runs.
- Report p50/p95, not the best run.
- Measure resource metrics alongside runtime.
- Document the method so results can be reproduced.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Run a controlled comparison

**Steps**

1. Choose one query and two partition counts.
2. Run each configuration three times, alternating order, after a warmup.
3. Record runtime, shuffle, spill, and GC; compare medians and spread.

**What to observe**: Run-to-run variance can be large; repeated measurements reveal whether the difference is real.

**Healthy result**: The improvement is consistent across repetitions.

**Problematic result**: The two configurations overlap, so the change is inconclusive.

**Engineering takeaway**: A benchmark without variance is an anecdote.

---

## Trade-offs and Alternatives

### Advantages

- Defensible results
- Comparable configurations
- Supports cost decisions

### Limitations

- Time-consuming
- Synthetic data may not represent production
- Cluster noise

### When to use it

- Tuning validation
- Platform comparisons
- Capacity planning

### When not to use it

- Single-run claims
- Cherry-picked numbers
- Changing multiple variables

### Alternatives inside Spark

- Spark UI
- Event logs
- Query plans

### Alternatives outside Spark

- Benchmark suites
- APM tools
- Cost calculators

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Benchmark Join
**Problem Statement**: Benchmark join between orders and customers.  
**Input / Context**: orders, customers

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Measure Shuffle Read
**Problem Statement**: How to inspect shuffle read metrics via statusTracker.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): A/B Testing Framework
**Problem Statement**: Design a function to compare two DataFrames execution times.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The benchmark that only measured the cache

**Scenario**: The second configuration looks much faster because it reused a warmed cache from the first run.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Clear caches or restart the session between configurations.
2. Randomize or alternate the run order.
3. Measure cold and warm runs separately.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Benchmark Join):
```python
custs = spark.read.option('header', True).csv('/opt/spark-data/retail/customers.csv')
benchmark_op('Join', lambda: orders.join(custs,'customer_id').count())
```

#### Solution 2 (Measure Shuffle Read):
```python
print('Query TaskContext or statusTracker for stage metrics.')
```

#### Solution 3 (A/B Testing Framework):
```python
print('Run both functions with multiple iterations and statistical t-test.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Warmup Rule**: Always warm up the JVM before recording metrics.
- **Action Isolation**: Measure wall clock exclusively around the terminal Action.

### Essential Technical Terms
- `JIT Compilation`: JVM runtime bytecode compilation to native machine code.
- `Cold Run`: First execution before caches/compilation settle.

### Common Engineering Mistakes
- **Measuring transformation lines**: Transformations are lazy; timing them measures AST creation, not compute.

### Interview Questions on This Topic
- **Q: Why is the first execution of a Spark query slower than subsequent runs?**
  - *A*: Because of initial classloading, JVM JIT compilation, executor thread initialization, and storage metadata caching.


### More Interview Questions

- **Q: Why discard warmup runs?**
  - *A*: The JVM, caches, and I/O may not be in steady state during the first run, so including it can bias the result.
- **Q: Why report percentiles instead of the fastest run?**
  - *A*: Percentiles capture variance and typical behavior; the fastest run is often an outlier and not representative of production.

### What to Study Next
Module 10 complete! Proceed to Module 11: Fault Tolerance and Reliability.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')